# Claude Code Configuration & Workflows

## Table of contents
- [Setup](#setup)
- [Feature 1: CLAUDE.md — Persistent Memory & Instruction Hierarchy](#feature-1-claudemd)
- [Feature 2: Settings Configuration](#feature-2-settings-configuration)
- [Feature 3: Plan Mode](#feature-3-plan-mode)
- [Feature 4: Custom Slash Commands](#feature-4-custom-slash-commands)
- [Feature 5: Hooks — Deterministic Lifecycle Actions](#feature-5-hooks)
- [Feature 6: Non-interactive CLI (--print & --output-format)](#feature-6-non-interactive-cli)
- [Feature 7: CI/CD Integration](#feature-7-cicd-integration)
- [Summary](#summary)

## Introduction

Claude Code is a CLI tool that runs Claude as an agentic coding assistant. Beyond running `claude`
interactively, it exposes a rich configuration system that lets you:

- Inject persistent context through **CLAUDE.md** files across multiple hierarchy levels
- Control tool permissions and behavior via **settings files**
- Enable a **plan-before-execute** workflow with Plan Mode
- Define **reusable prompt templates** as slash commands
- Enforce **guardrails and audit trails** through lifecycle hooks
- Drive **headless automation** via `--print` and `--output-format json`
- **Integrate with CI/CD pipelines** for automated code review and generation

This notebook walks through each configuration layer with TypeScript examples.


## Setup

Most cells in this notebook demonstrate configuration patterns and **do not** require an active
API connection — they define types, show file structures, and print examples.

Cells marked `// requires: claude CLI` need the `claude` command installed and
`ANTHROPIC_API_KEY` set in the environment.

In [ ]:
import { load } from 'https://deno.land/std@0.220.0/dotenv/mod.ts';

await load({ envPath: '../.env', export: true });

const prettyJson = (obj: unknown): string => JSON.stringify(obj, null, 2);

console.log('Setup complete.');

## Feature 1: CLAUDE.md

**What**: `CLAUDE.md` is a Markdown file that Claude Code automatically reads at startup and
injects as persistent context into every session.

**Why**: Instead of repeating project context in every prompt, define it once. CLAUDE.md is
the onboarding document for the agent — code standards, tool locations, domain constraints,
architectural decisions, and team conventions all belong here.

**What CLAUDE.md provides vs. what it does not**:

| Provides | Does not provide |
|---|---|
| Background context and strategy | Hard constraints on data sources |
| Coding standards and conventions | Guaranteed override of live data |
| Tool and script locations | Execution permissions (use settings for that) |
| Domain knowledge the agent can draw on | Security boundaries |

> **Key insight**: When both a CLAUDE.md and detailed data files (CSVs, code) are present,
> the agent naturally seeks the most authoritative source. CLAUDE.md provides *orientation*;
> use explicit prompt instructions to direct which source to prefer.

### CLAUDE.md Hierarchy

Claude Code reads CLAUDE.md files from multiple locations and **stacks** them — all layers
are loaded simultaneously, with higher-precedence layers winning on conflicts.

| Level | Location | Scope | Committed to git? |
|---|---|---|---|
| **Enterprise** | Org-managed policy | All org members | N/A (admin-controlled) |
| **User** | `~/.claude/CLAUDE.md` | All projects for this user | No (personal) |
| **Workspace** | `<workspace-root>/CLAUDE.md` | All sub-projects in the workspace | Yes (if shared) |
| **Project** | `./CLAUDE.md` or `./.claude/CLAUDE.md` | This project only | Yes |
| **Sub-directory** | `./src/CLAUDE.md`, `./tests/CLAUDE.md` | Files under that directory | Yes |

**Precedence** (highest → lowest): `Enterprise` → `User` → `Workspace` → `Project` → `Sub-directory`

```
~/
└── .claude/
    └── CLAUDE.md          ← User-level (personal; applies to ALL projects)

monorepo/                  ← Workspace root
├── CLAUDE.md              ← Workspace-level (applies to all sub-projects)
├── backend/
│   ├── CLAUDE.md          ← Project-level (backend-specific standards)
│   └── src/
│       └── CLAUDE.md      ← Sub-directory (src/ specific rules)
└── frontend/
    └── CLAUDE.md          ← Project-level (frontend-specific standards)
```

**Practical guidance**:
- **Project** CLAUDE.md: team-wide standards, commit it
- **User** `~/.claude/CLAUDE.md`: personal preferences (preferred style, shortcuts)
- **Workspace** CLAUDE.md: monorepo-wide shared context
- **Sub-directory** CLAUDE.md: domain-specific rules (e.g. `tests/` has different conventions)
- **Enterprise** CLAUDE.md: compliance/security rules that cannot be overridden by users

In [ ]:
// Show what each CLAUDE.md level typically contains

const claudeMdExamples: Record<string, string> = {
  'Project (./CLAUDE.md)': [
    '# TechStart Backend Service',
    '',
    '## Project Context',
    'B2B SaaS API built with Node.js + TypeScript. Deployed on AWS ECS.',
    'Database: PostgreSQL 15 (Prisma ORM).',
    '',
    '## Code Standards',
    '- All functions must have explicit TypeScript return types',
    '- Use async/await — never raw .then() chains',
    '- Max line length: 100 characters',
    '',
    '## Key Paths',
    '- Migrations: prisma/migrations/',
    '- API routes: src/routes/',
  ].join('\n'),

  'User (~/.claude/CLAUDE.md)': [
    '# Personal Claude Code Preferences',
    '',
    'I prefer verbose variable names over abbreviations.',
    'Always explain WHY a change is made, not just WHAT.',
    'When unsure about large structural changes, ask first.',
  ].join('\n'),

  'Sub-directory (tests/CLAUDE.md)': [
    '# Test Directory Standards',
    '',
    'Test files use Jest. Every test file must:',
    '- Import real database (no mocks) — integration tests only',
    '- Clean up after itself using afterEach()',
    '- Use descriptive test names that read like sentences',
  ].join('\n'),
};

for (const [level, content] of Object.entries(claudeMdExamples)) {
  console.log(`=== ${level} ===`);
  console.log(content);
  console.log();
}

// Hierarchy reminder
const hierarchy = [
  { level: 'Enterprise', path: '(org-managed policy)' },
  { level: 'User',       path: '~/.claude/CLAUDE.md' },
  { level: 'Workspace',  path: '<workspace-root>/CLAUDE.md' },
  { level: 'Project',    path: './CLAUDE.md or ./.claude/CLAUDE.md' },
  { level: 'Sub-dir',   path: './src/CLAUDE.md (etc.)' },
];

console.log('Precedence (highest → lowest):');
hierarchy.forEach((h, i) => console.log(`  ${i + 1}. [${h.level}] ${h.path}`));

## Feature 2: Settings Configuration

Claude Code reads three settings files that control **permissions, tools, and behavior**.
Unlike CLAUDE.md (which provides context), settings files enforce **hard configuration**:
which tools are allowed, what commands can run, and which hooks fire.

| File | Location | Git-committed? | Purpose |
|---|---|---|---|
| `settings.json` | `.claude/settings.json` | ✅ Yes | Team-shared project settings |
| `settings.local.json` | `.claude/settings.local.json` | ❌ No (gitignore) | Local-only: hooks, personal overrides |
| User settings | `~/.claude/settings.json` | ❌ No | Global user preferences |

**Precedence**: User → `settings.local.json` → `settings.json`

### Key Settings Fields

```json
{
  "allowedTools": ["Bash", "Read", "Write"],
  "deniedTools": ["WebSearch"],
  "permissions": {
    "allow": ["Bash(git *)", "Bash(npm test)"],
    "deny":  ["Bash(rm -rf *)"]
  },
  "hooks": { ... },
  "outputStyle": "executive"
}
```

> **`allowedTools` vs `permissions.allow`**:  
> `allowedTools` is a coarse on/off toggle per tool name.  
> `permissions.allow` / `permissions.deny` supports patterns — allow `Bash(git *)` while denying `Bash(rm -rf *)`.

### `setting_sources` (Agent SDK)

When using the Agent SDK, the `setting_sources` parameter controls which files are loaded:

| Value | File loaded |
|---|---|
| `'project'` | `.claude/settings.json` |
| `'local'` | `.claude/settings.local.json` |
| `'user'` | `~/.claude/settings.json` |

Hooks live in `settings.local.json` — always include **both** `'project'` and `'local'`
when hooks must fire. Slash commands and CLAUDE.md only need `'project'`.

In [ ]:
// TypeScript interfaces matching the Claude Code settings schema

interface HookEntry {
  type: 'command';
  command: string;  // path to executable, may use $CLAUDE_PROJECT_DIR
}

interface HookMatcher {
  matcher: string;       // tool name, e.g. 'Write', 'Edit', 'Bash'
  hooks: HookEntry[];
}

interface ClaudePermissions {
  allow?: string[];   // e.g. ['Bash(npm test)', 'Bash(git *)']
  deny?: string[];    // e.g. ['Bash(rm -rf *)', 'Bash(git push --force)']
}

interface ClaudeSettings {
  allowedTools?: string[];
  deniedTools?: string[];
  permissions?: ClaudePermissions;
  hooks?: {
    PreToolUse?: HookMatcher[];
    PostToolUse?: HookMatcher[];
    Stop?: HookMatcher[];
  };
  outputStyle?: string;
}

// Committed to git: team-wide project settings (minimal, permissive for dev)
const projectSettings: ClaudeSettings = {
  allowedTools: ['Read', 'Write', 'Edit', 'Bash', 'WebSearch'],
  permissions: {
    deny: ['Bash(git push --force *)', 'Bash(DROP TABLE *)'],
  },
};

// Gitignored: local settings with hooks for audit trail
const localSettings: ClaudeSettings = {
  hooks: {
    PostToolUse: [
      {
        matcher: 'Write',
        hooks: [{ type: 'command', command: '$CLAUDE_PROJECT_DIR/.claude/hooks/audit.sh' }],
      },
    ],
  },
};

// CI-specific: read-only, tightly scoped (would be in a separate ci-settings.json)
const ciSettings: ClaudeSettings = {
  allowedTools: ['Read', 'Bash'],
  permissions: {
    allow: ['Bash(npm test)', 'Bash(npm run lint)', 'Bash(git diff *)', 'Bash(git log *)'],
    deny:  ['Bash(git push *)', 'Bash(rm *)', 'Bash(curl *)', 'Bash(npm publish *)'],
  },
};

console.log('settings.json (project, committed):');
console.log(prettyJson(projectSettings));
console.log('\nsettings.local.json (gitignored, local):');
console.log(prettyJson(localSettings));
console.log('\nCI settings:');
console.log(prettyJson(ciSettings));

## Feature 3: Plan Mode

**What**: Plan Mode instructs Claude Code to produce a detailed execution plan **without taking
any action**. The agent analyzes requirements and proposes steps, but does not modify files,
run commands, or make network requests.

**Why**: Complex or high-stakes tasks benefit from a *review-before-execute* cycle:
1. Generate plan → human reviews → approve → execute
2. Reduces costly mistakes from misunderstood requirements
3. Creates an audit trail for compliance

**How to invoke**:

| Context | Flag / Parameter |
|---|---|
| CLI (interactive) | `claude --permission-mode plan` |
| CLI (non-interactive) | `claude --print --permission-mode plan "<prompt>"` |
| Agent SDK | `permission_mode: 'plan'` in `ClaudeAgentOptions` |

**Plan → Approve → Execute workflow**:
```bash
# Step 1: Generate plan (no file changes)
claude --print --permission-mode plan "Refactor the auth module to use JWT"

# Step 2: Human reviews the plan output

# Step 3: Execute (files changed now)
claude --print "Execute the plan: refactor the auth module to use JWT"
```

> **Note**: The agent calls `ExitPlanMode()` internally when done planning. In the interactive
> CLI, this transitions to execution mode. In headless scripts, start a new invocation without
> `--permission-mode plan` to execute.

In [ ]:
// requires: claude CLI
// Pattern for invoking Plan Mode from TypeScript and saving the plan

interface ClaudeJsonOutput {
  type: string;
  subtype: 'success' | 'error';
  result: string;
  session_id: string;
  is_error: boolean;
  duration_ms: number;
  num_turns: number;
  total_cost_usd: number;
}

async function generatePlan(prompt: string): Promise<{ plan: string; sessionId: string }> {
  const cmd = new Deno.Command('claude', {
    args: [
      '--print',
      '--output-format', 'json',
      '--permission-mode', 'plan',
      '--max-turns', '5',
      prompt,
    ],
    stdout: 'piped',
    stderr: 'piped',
    env: { ANTHROPIC_API_KEY: Deno.env.get('ANTHROPIC_API_KEY') ?? '' },
  });

  const { code, stdout, stderr } = await cmd.output();
  if (code !== 0) {
    throw new Error(`plan generation failed: ${new TextDecoder().decode(stderr)}`);
  }

  const out = JSON.parse(new TextDecoder().decode(stdout)) as ClaudeJsonOutput;
  return { plan: out.result, sessionId: out.session_id };
}

// Review-then-execute helper
async function reviewAndExecute(
  task: string,
  approve: (plan: string) => Promise<boolean>,
): Promise<ClaudeJsonOutput | null> {
  const { plan } = await generatePlan(task);
  console.log('=== Generated Plan ===\n', plan, '\n');

  const approved = await approve(plan);
  if (!approved) {
    console.log('Plan rejected — no changes made.');
    return null;
  }

  // Execute: same task, no plan mode
  const cmd = new Deno.Command('claude', {
    args: ['--print', '--output-format', 'json', task],
    stdout: 'piped',
    stderr: 'piped',
    env: { ANTHROPIC_API_KEY: Deno.env.get('ANTHROPIC_API_KEY') ?? '' },
  });
  const { code, stdout } = await cmd.output();
  if (code !== 0) throw new Error('execution failed');
  return JSON.parse(new TextDecoder().decode(stdout)) as ClaudeJsonOutput;
}

console.log('generatePlan() and reviewAndExecute() helpers defined.');
console.log('Usage:');
console.log('  const { plan } = await generatePlan("Refactor auth module to use JWT");');
console.log('  // review plan...');
console.log('  await reviewAndExecute(task, async (plan) => userApproves(plan));');

## Feature 4: Custom Slash Commands

**What**: Slash commands are **reusable prompt templates** stored as Markdown files. Users
trigger them with `/command-name [arguments]` shorthand in Claude Code.

**Why**: Recurring queries deserve well-crafted, consistent prompts. Slash commands encode
institutional prompt knowledge and reduce inconsistency across team members.

**How** — create `.claude/commands/<name>.md`:
```markdown
---
name: pr-review
description: Review a pull request diff for code quality issues
---

Review the following pull request for:
1. Adherence to project coding standards (see CLAUDE.md)
2. Potential bugs or edge cases
3. Missing tests
4. Security issues

PR branch or diff: $ARGUMENTS
```

**Argument placeholders**:
| Placeholder | Meaning |
|---|---|
| `$ARGUMENTS` | Full argument string the user typed |
| `$1`, `$2` ... | Positional arguments |

**Usage**: `/pr-review feature/auth-refactor` expands to the full prompt with `feature/auth-refactor` substituted for `$ARGUMENTS`.

> **SDK note**: Slash commands require `setting_sources: ['project']` in `ClaudeAgentOptions`.
> By default, the SDK operates in isolation and does NOT load filesystem settings.

In [ ]:
// TypeScript: programmatically generate slash command files

interface SlashCommand {
  name: string;
  description: string;
  promptTemplate: string;  // may contain $ARGUMENTS, $1, $2 ...
}

function renderCommandFile(cmd: SlashCommand): string {
  return `---\nname: ${cmd.name}\ndescription: ${cmd.description}\n---\n\n${cmd.promptTemplate}\n`;
}

const projectCommands: SlashCommand[] = [
  {
    name: 'run-tests',
    description: 'Run the test suite and summarize failures',
    promptTemplate:
      'Run `npm test` and provide a concise summary:\n' +
      '1. How many tests passed / failed\n' +
      '2. Root cause of each failure\n' +
      '3. Suggested fix for the first three failures\n',
  },
  {
    name: 'explain-file',
    description: 'Explain what a source file does and how it fits the architecture',
    promptTemplate:
      'Read $ARGUMENTS and explain:\n' +
      '1. What this file does (2–3 sentences)\n' +
      '2. Its main exports / classes / functions\n' +
      '3. How it connects to the rest of the codebase\n',
  },
  {
    name: 'security-scan',
    description: 'Scan a file or directory for common security issues',
    promptTemplate:
      'Perform a security review of $ARGUMENTS. Check for:\n' +
      '- SQL injection, XSS, command injection\n' +
      '- Hardcoded secrets or credentials\n' +
      '- Insecure dependencies\n' +
      '- OWASP Top 10 patterns\n' +
      'Output a ranked list with severity (high/medium/low) and remediation.\n',
  },
];

// Show the generated file content for each command
for (const cmd of projectCommands) {
  console.log(`=== .claude/commands/${cmd.name}.md ===`);
  console.log(renderCommandFile(cmd));
}

// Invocation examples
console.log('=== Usage examples ===');
console.log('/run-tests                     → runs tests, summarizes results');
console.log('/explain-file src/routes/auth.ts → explains that specific file');
console.log('/security-scan src/             → scans entire src directory');

## Feature 5: Hooks

**What**: Hooks are executables (shell scripts, TypeScript files, Python scripts) that run
**deterministically** before or after specific tool calls. Configured in
`.claude/settings.local.json`.

**Why**: Unlike prompts (probabilistic), hooks are **guaranteed to run**. Use them for:
- Audit trails and compliance logging
- Blocking dangerous operations (exit non-zero to abort the tool call)
- Notifications, metrics, or external system updates

**Hook events**:

| Event | When it fires | Can block the tool call? |
|---|---|---|
| `PreToolUse` | Before the tool executes | ✅ Yes — exit non-zero |
| `PostToolUse` | After the tool completes | ❌ No (observational only) |
| `Stop` | When the agent session ends | ❌ No |

**Environment variables available inside hook scripts**:
| Variable | Value |
|---|---|
| `$CLAUDE_TOOL_NAME` | Name of the tool being called (e.g. `Write`) |
| `$CLAUDE_TOOL_INPUT` | JSON-encoded tool input |
| `$CLAUDE_PROJECT_DIR` | Absolute path of the project root |
| `$CLAUDE_SESSION_ID` | Unique ID for this Claude Code session |

**Configuration in `.claude/settings.local.json`** (gitignored):
```json
{
  "hooks": {
    "PostToolUse": [
      { "matcher": "Write", "hooks": [{ "type": "command", "command": ".claude/hooks/audit.ts" }] }
    ],
    "PreToolUse": [
      { "matcher": "Bash",  "hooks": [{ "type": "command", "command": ".claude/hooks/guard.ts" }] }
    ]
  }
}
```

In [ ]:
// TypeScript hook script examples
// Hooks can be any executable — here they use Deno for TypeScript execution.

// Hook 1 (PostToolUse): append an audit log entry on Write / Edit
const auditLogHook = `#!/usr/bin/env -S deno run --allow-env --allow-write
// PostToolUse hook: logs every Write/Edit operation to audit.jsonl

const toolName = Deno.env.get('CLAUDE_TOOL_NAME') ?? 'unknown';
const toolInput = Deno.env.get('CLAUDE_TOOL_INPUT') ?? '{}';
const sessionId = Deno.env.get('CLAUDE_SESSION_ID') ?? 'unknown';
const projectDir = Deno.env.get('CLAUDE_PROJECT_DIR') ?? '.';

const entry = JSON.stringify({
  timestamp: new Date().toISOString(),
  sessionId,
  tool: toolName,
  input: JSON.parse(toolInput),
}) + '\\n';

await Deno.writeTextFile(
  projectDir + '/.claude/audit.jsonl',
  entry,
  { append: true },
);
// Exit 0 implicitly — PostToolUse hooks cannot block
`;

// Hook 2 (PreToolUse): block dangerous shell patterns
const guardBashHook = `#!/usr/bin/env -S deno run --allow-env
// PreToolUse hook: blocks dangerous Bash patterns.
// Exiting with non-zero code ABORTS the tool call.

const toolInput = Deno.env.get('CLAUDE_TOOL_INPUT') ?? '{}';
const { command } = JSON.parse(toolInput) as { command: string };

const BLOCKED: RegExp[] = [
  /rm\\s+-rf/,
  /git push.*--force/,
  /DROP TABLE/i,
  /curl.*\\|.*sh/,    // curl-pipe-bash supply-chain pattern
];

for (const pattern of BLOCKED) {
  if (pattern.test(command)) {
    console.error('BLOCKED by hook:', command);
    Deno.exit(1);  // non-zero = tool call is aborted
  }
}
// Exit 0 = allow the tool call
Deno.exit(0);
`;

console.log('=== .claude/hooks/audit-log.ts ===');
console.log(auditLogHook);
console.log('=== .claude/hooks/guard-bash.ts ===');
console.log(guardBashHook);

In [ ]:
// Generate the settings.local.json that wires the hooks defined above

const settingsLocal: ClaudeSettings = {
  hooks: {
    PostToolUse: [
      {
        matcher: 'Write',
        hooks: [{ type: 'command', command: '$CLAUDE_PROJECT_DIR/.claude/hooks/audit-log.ts' }],
      },
      {
        matcher: 'Edit',
        hooks: [{ type: 'command', command: '$CLAUDE_PROJECT_DIR/.claude/hooks/audit-log.ts' }],
      },
    ],
    PreToolUse: [
      {
        matcher: 'Bash',
        hooks: [{ type: 'command', command: '$CLAUDE_PROJECT_DIR/.claude/hooks/guard-bash.ts' }],
      },
    ],
  },
};

console.log('.claude/settings.local.json:');
console.log(prettyJson(settingsLocal));

console.log('\nKey rules:');
console.log('  • settings.local.json is gitignored — hooks do not run in CI by default');
console.log('  • PreToolUse hooks BLOCK a call when they exit with non-zero code');
console.log('  • PostToolUse hooks are observational — they cannot undo a completed call');
console.log('  • Use setting_sources: ["project", "local"] in the SDK to load hooks');

## Feature 6: Non-interactive CLI (`--print` and `--output-format`)

**What**: By default, `claude` runs interactively (REPL-style). The `--print` flag switches
to **non-interactive (headless) mode** — Claude processes the prompt and exits immediately.

**Why**: Enables scripting, piping, CI/CD automation, and programmatic integration.

### `--print`

```bash
# Process a prompt and exit
claude --print "Summarize the last 5 git commits"

# Pipe file content in
cat README.md | claude --print "Identify missing sections"

# Pipe output to another tool
claude --print "List all TODO comments in src/" | grep -i 'security'
```

### `--output-format`

| Format | Description | Best for |
|---|---|---|
| `text` (default) | Plain text response | Human-readable output |
| `json` | Full structured object | Programmatic processing |
| `stream-json` | NDJSON event stream | Real-time / streaming |

> **`--print` is required for `--output-format`** — structured output only works in
> non-interactive mode.

### JSON output schema (`--output-format json`)

```json
{
  "type": "result",
  "subtype": "success",
  "result": "<Claude's response text>",
  "session_id": "sess_abc123",
  "is_error": false,
  "duration_ms": 1842,
  "num_turns": 2,
  "total_cost_usd": 0.0023
}
```

**`is_error`** lets you detect failures without parsing stderr — a reliable contract for
automation. **`total_cost_usd`** enables per-task cost tracking in pipelines.

In [ ]:
// TypeScript wrapper for the Claude CLI in non-interactive mode

interface ClaudeCliOptions {
  outputFormat?: 'text' | 'json' | 'stream-json';
  permissionMode?: 'default' | 'plan' | 'bypassPermissions';
  allowedTools?: string[];
  maxTurns?: number;
  systemPrompt?: string;
  cwd?: string;
}

// requires: claude CLI
async function runClaude(
  prompt: string,
  options: ClaudeCliOptions = {},
): Promise<ClaudeJsonOutput | string> {
  const { outputFormat = 'json', permissionMode, allowedTools, maxTurns, systemPrompt, cwd } = options;

  const args: string[] = ['--print', '--output-format', outputFormat];
  if (permissionMode && permissionMode !== 'default') args.push('--permission-mode', permissionMode);
  if (maxTurns !== undefined) args.push('--max-turns', String(maxTurns));
  if (systemPrompt) args.push('--system-prompt', systemPrompt);
  if (allowedTools?.length) args.push('--allowedTools', allowedTools.join(','));
  args.push(prompt);

  const cmd = new Deno.Command('claude', {
    args,
    cwd,
    stdout: 'piped',
    stderr: 'piped',
    env: { ANTHROPIC_API_KEY: Deno.env.get('ANTHROPIC_API_KEY') ?? '' },
  });

  const { code, stdout, stderr } = await cmd.output();
  if (code !== 0) {
    throw new Error(`claude CLI error (exit ${code}): ${new TextDecoder().decode(stderr)}`);
  }

  const raw = new TextDecoder().decode(stdout);
  return outputFormat === 'json' ? JSON.parse(raw) as ClaudeJsonOutput : raw;
}

console.log('runClaude() wrapper ready.');
console.log();
console.log('Examples:');
console.log('  const out = await runClaude("What is 2+2?", { maxTurns: 1 });');
console.log('  // out.result   → "4"');
console.log('  // out.is_error → false');
console.log('  // out.total_cost_usd → 0.0003');
console.log();
console.log('  const plan = await runClaude(task, { permissionMode: "plan" });');
console.log('  // plan.result contains the plan text; no files were changed');

In [ ]:
// Simulate and parse --output-format json output

const mockSuccessOutput: ClaudeJsonOutput = {
  type: 'result',
  subtype: 'success',
  result: 'The function `calculateTax` has a bug on line 47: integer division truncates\n' +
          'decimals for amounts not divisible by 100. Fix: use floating-point division.',
  session_id: 'sess_demo_abc123',
  is_error: false,
  duration_ms: 2341,
  num_turns: 2,
  total_cost_usd: 0.0031,
};

const mockErrorOutput: ClaudeJsonOutput = {
  type: 'result',
  subtype: 'error',
  result: 'Could not read file: src/auth.ts not found',
  session_id: 'sess_demo_err456',
  is_error: true,
  duration_ms: 450,
  num_turns: 1,
  total_cost_usd: 0.0004,
};

function processOutput(output: ClaudeJsonOutput): void {
  if (output.is_error) {
    console.error(`[ERROR] ${output.result}`);
    return;
  }
  console.log(`[OK] ${output.result}`);
  console.log(`     turns=${output.num_turns}  ms=${output.duration_ms}  cost=$${output.total_cost_usd.toFixed(4)}`);
}

console.log('Success case:');
processOutput(mockSuccessOutput);
console.log();
console.log('Error case:');
processOutput(mockErrorOutput);

console.log('\nKey --output-format json advantages:');
console.log('  is_error     — detect failures without parsing stderr');
console.log('  total_cost_usd — per-task cost tracking in pipelines');
console.log('  num_turns    — detect runaway agent loops');
console.log('  session_id   — resume or correlate multi-step workflows');

## Feature 7: CI/CD Integration

Claude Code operates fully headless in CI/CD pipelines using `--print` and environment
variables — no interactive session required.

### Common CI/CD use cases

| Use case | CLI pattern |
|---|---|
| Automated PR review | `claude --print "Review this diff: $(git diff main)"` |
| Code quality gate | `claude --print --output-format json "Find issues in src/"` |
| Test failure analysis | `npm test 2>&1 \| claude --print "Summarize failures"` |
| Documentation generation | `claude --print "Document all public APIs in src/api/"` |
| Dry-run security scan | `claude --print --permission-mode plan "Check for OWASP Top 10"` |

### GitHub Actions example

```yaml
name: Claude Code Review
on:
  pull_request:
    types: [opened, synchronize]

jobs:
  review:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0

      - name: Install Claude Code
        run: npm install -g @anthropic-ai/claude-code

      - name: Run Claude review
        env:
          ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
        run: |
          DIFF=$(git diff ${{ github.event.pull_request.base.sha }} HEAD)
          claude --print --output-format json \
            "Review this PR diff for bugs, missing tests, and security issues: $DIFF" \
            > review.json
          # Fail CI if Claude found critical issues
          IS_ERROR=$(jq -r '.is_error' review.json)
          [ "$IS_ERROR" = 'false' ] || exit 1
          jq -r '.result' review.json
```

### Security checklist for CI/CD

1. **Never commit `ANTHROPIC_API_KEY`** — use CI secrets and environment variables
2. **Use `--permission-mode plan`** for review tasks — prevents accidental file changes in CI
3. **Scope tools tightly** via `settings.json`: only allow `Read`, `Bash(npm test)`, etc.
4. **Deny write operations**: `Bash(git push *)`, `Bash(rm *)`, `Bash(npm publish *)`
5. **Pin Claude Code version**: `npm install -g @anthropic-ai/claude-code@<version>` for reproducibility

In [ ]:
// TypeScript CI/CD helper: invoke Claude, parse result, and decide gate outcome

interface CiGateResult {
  passed: boolean;
  issues: string[];
  summary: string;
  costUsd: number;
}

// requires: claude CLI
async function runCiCodeReview(diffContent: string): Promise<CiGateResult> {
  const prompt =
    'Review this code diff. Respond ONLY with a JSON object (no markdown) matching:\n' +
    '{ passed: boolean, issues: string[], summary: string }\n' +
    'Set passed=false if any critical or high-severity issue is found.\n\n' +
    'Diff:\n' + diffContent;

  const output = await runClaude(prompt, {
    outputFormat: 'json',
    permissionMode: 'plan',    // read-only in CI
    allowedTools: ['Read'],
    maxTurns: 3,
  }) as ClaudeJsonOutput;

  if (output.is_error) {
    return { passed: false, issues: [output.result], summary: 'Review failed', costUsd: 0 };
  }

  const review = JSON.parse(output.result) as Omit<CiGateResult, 'costUsd'>;
  return { ...review, costUsd: output.total_cost_usd };
}

// Simulate the CI outcome
const mockGateResult: CiGateResult = {
  passed: false,
  issues: [
    'Missing null check on line 23 of auth.ts — can throw at runtime',
    'resetPassword() has no unit test coverage',
  ],
  summary: 'PR has 2 issues that must be addressed before merge.',
  costUsd: 0.0087,
};

console.log('CI gate result:');
console.log(prettyJson(mockGateResult));

if (!mockGateResult.passed) {
  console.log(`\n\u274c CI gate FAILED — ${mockGateResult.issues.length} issue(s)`);
  mockGateResult.issues.forEach((issue, i) => console.log(`   ${i + 1}. ${issue}`));
  // In a real CI script: Deno.exit(1);
} else {
  console.log('\n\u2705 CI gate PASSED');
}

In [ ]:
// Environment variable reference and CI settings.json recommendation

const claudeEnvVars: Record<string, string> = {
  ANTHROPIC_API_KEY:  '(required) API key — always store as a CI secret, never commit',
  CLAUDE_MODEL:       '(optional) Override model, e.g. claude-sonnet-4-6',
  ANTHROPIC_BASE_URL: '(optional) Custom endpoint for VPC/proxy routing',
};

console.log('Environment variables for CI/CD:');
for (const [name, desc] of Object.entries(claudeEnvVars)) {
  console.log(`  ${name.padEnd(25)} ${desc}`);
}

const ciSettingsRecommended: ClaudeSettings = {
  allowedTools: ['Read', 'Bash'],
  permissions: {
    allow: [
      'Bash(npm test)', 'Bash(npm run lint)',
      'Bash(git diff *)', 'Bash(git log *)', 'Bash(git status)',
    ],
    deny: [
      'Bash(git push *)', 'Bash(rm *)',
      'Bash(curl *)', 'Bash(npm publish *)',
    ],
  },
};

console.log('\nRecommended .claude/settings.json for CI:');
console.log(prettyJson(ciSettingsRecommended));

## Summary

| Feature | Config location | Key point |
|---|---|---|
| **CLAUDE.md** | `./CLAUDE.md`, `~/.claude/CLAUDE.md`, workspace root, enterprise | Hierarchy: Enterprise > User > Workspace > Project > Sub-dir |
| **Settings** | `.claude/settings.json` / `settings.local.json` | `local` is gitignored; required for hooks |
| **Plan Mode** | CLI: `--permission-mode plan` | No file changes; enables review-before-execute |
| **Slash Commands** | `.claude/commands/*.md` | Frontmatter + `$ARGUMENTS`; need `setting_sources: ['project']` in SDK |
| **Hooks** | Scripts + `settings.local.json` | `PreToolUse` can block (exit non-zero); `PostToolUse` is observational |
| **`--print`** | CLI flag | Required for all scripting and CI use |
| **`--output-format json`** | CLI flag | Requires `--print`; provides `is_error`, `total_cost_usd`, `session_id` |
| **CI/CD** | `settings.json` + CI secrets | Scope tools tightly; use `plan` mode for review tasks |

### Decision guide

```
Need to inject persistent context?           → CLAUDE.md (choose the right hierarchy level)
Need to restrict which tools Claude uses?     → settings.json (allowedTools / permissions)
Need review before execution?                 → --permission-mode plan
Need reusable prompt shortcuts?               → Slash commands (.claude/commands/)
Need guaranteed pre/post actions?             → Hooks (settings.local.json)
Need to run Claude in a script?               → --print [--output-format json]
Need Claude in CI/CD?                         → --print + CI secrets + scoped settings.json
```

### CLAUDE.md hierarchy at a glance

```
Enterprise (highest — org-managed, cannot be overridden)
    ↓
User  (~/.claude/CLAUDE.md — personal, cross-project)
    ↓
Workspace  (<root>/CLAUDE.md — monorepo-wide shared context)
    ↓
Project  (./CLAUDE.md — team standards, commit to git)
    ↓
Sub-directory  (./src/CLAUDE.md — directory-specific rules)
```